# Student Information

**Full Name:** [Your Full Name]

**Student ID:** 21201063

**Section:** [Your Section]

# Import Libraries

In [2]:
# Import required libraries
import random

print("Libraries imported successfully.")

Libraries imported successfully.


# Player Class

In [3]:
# ============================================================
# Player Class
# This class represents a single player in the game.
# ============================================================

class Player:
    """A simple class to store player information."""

    def __init__(self, name, role):
        """Initialize a player with a name and role."""
        self.name = name
        self.role = role
        self.alive = True

    def display_info(self):
        """Show the player's name, role and status."""
        status = "Alive"
        if self.alive == False:
            status = "Eliminated"
        print(self.name + " - " + self.role + " (" + status + ")")

    def eliminate(self):
        """Mark this player as eliminated."""
        self.alive = False
        print(self.name + " has been eliminated.")

# Game Class

In [4]:
# ============================================================
# Game Class
# This class controls the whole game flow.
# ============================================================

class Game:
    """Main game controller for the Werewolf game."""

    def __init__(self):
        """Set up the game with empty players list and round 0."""
        self.players = []
        self.round_number = 0
        self.eliminated_order = []

    # ----------------------------------------------------------
    # File reading: load player names from players.txt
    # ----------------------------------------------------------
    def load_players(self, filename):
        """Read player names from a file and create Player objects."""
        print("Loading players from " + filename + "...")

        # Open the file and read line by line
        file = open(filename, "r")
        for line in file:
            name = line.strip()
            if name != "":
                # Create a Player with no role yet
                player = Player(name, "Unassigned")
                self.players.append(player)
        file.close()

        print("Loaded " + str(len(self.players)) + " players.")

    # ----------------------------------------------------------
    # Role assignment: 2 Werewolves, rest are Villagers
    # ----------------------------------------------------------
    def assign_roles(self):
        """Randomly pick 2 players to be Werewolves, others become Villagers."""
        print("\nAssigning roles randomly...")

        # Make a copy of the players list and shuffle it
        shuffled = list(self.players)
        random.shuffle(shuffled)

        # First 2 players become werewolves
        for i in range(len(shuffled)):
            if i < 2:
                shuffled[i].role = "Werewolf"
            else:
                shuffled[i].role = "Villager"

        print("Roles have been assigned. (Shh... it's a secret!)")

    # ----------------------------------------------------------
    # Helper: get all alive players
    # ----------------------------------------------------------
    def get_alive_players(self):
        """Return a list of players who are still alive."""
        alive_list = []
        for player in self.players:
            if player.alive == True:
                alive_list.append(player)
        return alive_list

    # ----------------------------------------------------------
    # Helper: count alive players by role
    # ----------------------------------------------------------
    def count_alive_by_role(self, role):
        """Count how many alive players have a specific role."""
        count = 0
        for player in self.players:
            if player.alive == True and player.role == role:
                count = count + 1
        return count

    # ----------------------------------------------------------
    # Display: show all alive players
    # ----------------------------------------------------------
    def show_alive_players(self):
        """Print the list of all alive players."""
        alive = self.get_alive_players()
        print("\nAlive Players (" + str(len(alive)) + "):")
        for player in alive:
            print("  - " + player.name)

    # ----------------------------------------------------------
    # Night Phase: Werewolves attack a Villager
    # ----------------------------------------------------------
    def night_phase(self):
        """Handle the night phase: werewolves choose a victim."""

        print("\n" + "-" * 40)
        print("Night " + str(self.round_number))
        print("The village sleeps...")
        print("-" * 40)

        # Step 1: Find alive werewolves
        alive_wolves = []
        for player in self.players:
            if player.alive == True and player.role == "Werewolf":
                alive_wolves.append(player)

        # If no werewolves are alive, night is safe
        if len(alive_wolves) == 0:
            print("No werewolves are alive. The night is peaceful.")
            return

        # Step 2: Find alive villagers
        alive_villagers = []
        for player in self.players:
            if player.alive == True and player.role == "Villager":
                alive_villagers.append(player)

        # If no villagers to attack, skip
        if len(alive_villagers) == 0:
            print("No villagers left to attack.")
            return

        # Step 3: Pick a random villager to attack
        target = random.choice(alive_villagers)

        # Step 4: Check for Lucky Escape (10% chance)
        lucky_chance = random.randint(1, 100)
        if lucky_chance <= 10:
            # Lucky Escape event triggers!
            print("\n*** Special Event: Lucky Escape! ***")
            print("The target escaped from the Werewolves.")
            print("Nobody dies tonight.")
        else:
            # Normal attack
            print("\nWerewolves attacked " + target.name + ".")
            target.eliminate()
            self.eliminated_order.append(target.name)

    # ----------------------------------------------------------
    # Day Phase: Villagers vote to eliminate a suspect
    # ----------------------------------------------------------
    def day_phase(self):
        """Handle the day phase: villagers vote against someone."""

        print("\n" + "-" * 40)
        print("Day " + str(self.round_number))
        print("The village discusses...")
        print("-" * 40)

        alive = self.get_alive_players()

        # If only 1 or 0 alive, no voting needed
        if len(alive) <= 1:
            print("Not enough players for a vote.")
            return

        # Check for Mysterious Clue (15% chance)
        clue_chance = random.randint(1, 100)
        if clue_chance <= 15:
            # Find an alive werewolf to eliminate
            alive_wolves = []
            for player in alive:
                if player.role == "Werewolf":
                    alive_wolves.append(player)

            if len(alive_wolves) > 0:
                # Eliminate a werewolf directly
                caught_wolf = random.choice(alive_wolves)
                print("\n*** Special Event: Mysterious Clue! ***")
                print("The villagers found evidence and eliminated a Werewolf.")
                print("Villagers identified " + caught_wolf.name + " as a Werewolf.")
                caught_wolf.eliminate()
                self.eliminated_order.append(caught_wolf.name)
                return

        # Normal voting: randomly pick one alive player as suspect
        suspect = random.choice(alive)
        print("\nVillagers voted against " + suspect.name + ".")
        print(suspect.name + " was eliminated.")
        suspect.eliminate()
        self.eliminated_order.append(suspect.name)

    # ----------------------------------------------------------
    # Check Winner: See if the game should end
    # ----------------------------------------------------------
    def check_winner(self):
        """Check if either side has won the game.

        Returns:
            "Villagers" if villagers win
            "Werewolves" if werewolves win
            None if the game should continue
        """

        wolf_count = self.count_alive_by_role("Werewolf")
        villager_count = self.count_alive_by_role("Villager")

        # Villagers win when both werewolves are dead
        if wolf_count == 0:
            return "Villagers"

        # Werewolves win when they equal or outnumber villagers
        if wolf_count >= villager_count:
            return "Werewolves"

        # Game continues
        return None

    # ----------------------------------------------------------
    # Save Result: write game outcome to a file
    # ----------------------------------------------------------
    def save_result(self, winner):
        """Save the final game result to game_result.txt."""

        print("\nSaving results to game_result.txt...")

        # Gather information
        alive = self.get_alive_players()
        surviving_names = []
        for player in alive:
            surviving_names.append(player.name)

        # Write everything to the file
        file = open("game_result.txt", "w")
        file.write("=== WEREWOLF GAME RESULT ===\n")
        file.write("Winner: " + winner + "\n")
        file.write("Total Rounds: " + str(self.round_number) + "\n")
        file.write("\n--- Surviving Players ---\n")
        for name in surviving_names:
            file.write(name + " (" + self.get_role_by_name(name) + ")\n")
        file.write("\n--- Eliminated Players (in order) ---\n")
        for name in self.eliminated_order:
            file.write(name + "\n")
        file.close()

        print("Results saved successfully!")

    # ----------------------------------------------------------
    # Helper: get a player's role by name
    # ----------------------------------------------------------
    def get_role_by_name(self, name):
        """Find a player's role using their name."""
        for player in self.players:
            if player.name == name:
                return player.role
        return "Unknown"

    # ----------------------------------------------------------
    # Helper: print current status (villager vs werewolf count)
    # ----------------------------------------------------------
    def print_status(self):
        """Show how many villagers and werewolves are alive."""
        v_count = self.count_alive_by_role("Villager")
        w_count = self.count_alive_by_role("Werewolf")
        print("\nCurrent Status:")
        print("  Villagers: " + str(v_count))
        print("  Werewolves: " + str(w_count))

    # ----------------------------------------------------------
    # Start Game: the main game loop
    # ----------------------------------------------------------
    def start_game(self):
        """Run the main game loop until there is a winner."""

        print("\n" + "=" * 50)
        print("WEREWOLF GAME SIMULATOR")
        print("=" * 50)

        winner = None

        # Keep playing until someone wins
        while winner is None:
            self.round_number = self.round_number + 1

            print("\n" + "#" * 40)
            print("### Round " + str(self.round_number) + " ###")
            print("#" * 40)

            # Show current alive players
            self.show_alive_players()

            # Night phase
            self.night_phase()

            # Check if game ended during the night
            winner = self.check_winner()
            if winner is not None:
                break

            # Day phase
            self.day_phase()

            # Show status after the round
            self.print_status()

            # Check if game ended during the day
            winner = self.check_winner()

        # Game over - announce winner
        print("\n" + "=" * 50)
        print("GAME OVER!")
        print("Winner: " + winner.upper())
        print("=" * 50)

        # Save results
        self.save_result(winner)

        # Reveal all roles
        print("\n--- Final Role Reveal ---")
        for player in self.players:
            player.display_info()

# Create Players File

In [5]:
# ============================================================
# Create the players.txt file
# This file contains the names of all players, one per line.
# ============================================================

player_names = [
    "Alice",
    "Bob",
    "Charlie",
    "David",
    "Emma",
    "Frank",
    "Grace",
    "Henry"
]

# Write names to players.txt
file = open("players.txt", "w")
for name in player_names:
    file.write(name + "\n")
file.close()

print("players.txt has been created with " + str(len(player_names)) + " names.")

players.txt has been created with 8 names.


# Run Game Simulation

In [6]:
# ============================================================
# Run the Werewolf Game Simulation
# ============================================================

# Create the game object
game = Game()

# Load players from the file
game.load_players("players.txt")

# Assign werewolf/villager roles
game.assign_roles()

# Start the game loop
game.start_game()

Loading players from players.txt...
Loaded 8 players.

Assigning roles randomly...
Roles have been assigned. (Shh... it's a secret!)

WEREWOLF GAME SIMULATOR

########################################
### Round 1 ###
########################################

Alive Players (8):
  - Alice
  - Bob
  - Charlie
  - David
  - Emma
  - Frank
  - Grace
  - Henry

----------------------------------------
Night 1
The village sleeps...
----------------------------------------

Werewolves attacked Alice.
Alice has been eliminated.

----------------------------------------
Day 1
The village discusses...
----------------------------------------

Villagers voted against Emma.
Emma was eliminated.
Emma has been eliminated.

Current Status:
  Villagers: 5
  Werewolves: 1

########################################
### Round 2 ###
########################################

Alive Players (6):
  - Bob
  - Charlie
  - David
  - Frank
  - Grace
  - Henry

----------------------------------------
Night 2
The vil

# Result File Output

In [7]:
# ============================================================
# Display the contents of game_result.txt
# ============================================================

print("=" * 50)
print("CONTENTS OF game_result.txt")
print("=" * 50)

# Read and print the result file
result_file = open("game_result.txt", "r")
for line in result_file:
    print(line, end="")
result_file.close()

print("\n" + "=" * 50)

CONTENTS OF game_result.txt
=== WEREWOLF GAME RESULT ===
Winner: Werewolves
Total Rounds: 4

--- Surviving Players ---
Charlie (Villager)
Grace (Werewolf)

--- Eliminated Players (in order) ---
Alice
Emma
Frank
Henry
Bob
David



# Reflection Section

## Question 1: Which part was the hardest?

The hardest part for me was figuring out the game-ending condition. At first, I did not realize that Werewolves win when their number equals or exceeds the Villagers. I kept checking if all villagers were dead, but that is not the same thing. I had to read the assignment requirements carefully a few times. Once I understood the rule, writing the `check_winner()` method was straightforward.

## Question 2: What bug/problem did you face?

I ran into a problem where the game would crash during the night phase if no werewolves were alive. My code tried to pick a random villager to attack, but it also assumed there was always a werewolf making the attack. I fixed this by adding a check at the start of the `night_phase()` method. Now if there are no alive werewolves, the night phase just prints a message and returns safely.

## Question 3: What did you learn from this assignment?

This assignment taught me how to use classes in Python for a real project. Before this, I only wrote small programs without classes. I learned how to separate different parts of a program into methods. I also learned about file handling — reading from `players.txt` and writing results to `game_result.txt`. The random module was fun to use for picking targets and suspects. Overall, I now feel more comfortable with object-oriented programming.

## Question 4: What would you improve later?

If I had more time, I would add more roles to the game, like a Doctor who can save one person each night or a Seer who can check someone's role. I would also make the game interactive so the user can type commands instead of just watching the simulation run. Another improvement would be adding a graphical interface using a library like `tkinter`. I think these additions would make the project more interesting and fun to play.